# Defining subdomains for different materials
Goal: define subdomains by defining a discontinous cell-wise constant function 

In [1]:
from dolfinx import default_scalar_type
from dolfinx.fem import (
    Constant,
    dirichletbc,
    Function,
    functionspace,
    assemble_scalar,
    form,
    locate_dofs_geometrical,
    locate_dofs_topological,
)
from dolfinx.fem.petsc import LinearProblem
from dolfinx.io import XDMFFile, gmsh as gmshio
from dolfinx.mesh import create_unit_square, locate_entities
from dolfinx.plot import vtk_mesh

from ufl import SpatialCoordinate, TestFunction, TrialFunction, dx, grad, inner

from mpi4py import MPI

import meshio
import gmsh
import numpy as np
import pyvista

In [2]:
# Create mesh
mesh = create_unit_square(MPI.COMM_WORLD, 10, 10)
Q = functionspace(mesh, ("DG", 0)) # functionspace of discontinous cell-wise constant functions (one degree of freedom per cell)

Example: two materials. Domain $\Omega = [0,1]^2$ consists of two subdomains $\Omega_0 = [0,1]\times[0,0.5]$ and $\Omega_1 = [0,1]\times[0.5,1]$.

In [3]:
def Omega_0(x):
    return x[1] <= 0.5

def Omega_1(x):
    return x[1]>=0.5

Problem: Variable-coefficient extension of the Poisson equation

$-\nabla [\kappa(x,y)\nabla(x,y)]=1$ in $\Omega$ <br>
$u=u_D=1$ on $\partial\Omega_D=[0,y],~y\in[0,1]$<br>
$-\partial_n u = 0$ on $\partial\Omega \setminus \partial\Omega_D$ <br>
where $\kappa=1$ if $x\in\Omega_0$ and $\kappa=0.1$ if $x\in\Omega_1$

In [4]:
# Define kappa
kappa = Function(Q)
cells_0 = locate_entities(mesh, mesh.topology.dim, Omega_0) # find cells that are in Omega 0
cells_1 = locate_entities(mesh, mesh.topology.dim, Omega_1) # find cells that are in Omega 
kappa.x.array[cells_0] = np.full_like(cells_0, 1, dtype=default_scalar_type)
kappa.x.array[cells_1] = np.full_like(cells_1, 0.1, dtype=default_scalar_type)

In [5]:
# Alternative (kappa via interpolation)
def eval_kappa(x):
    values = np.zeros(x.shape[1], dtype=default_scalar_type)
    # Create a boolean array indicating which dofs (corresponding to cell centers) that are in each domain
    top_coords = x[1] > 0.5
    bottom_coords = x[1] < 0.5
    values[top_coords] = np.full(sum(top_coords), 0.1)
    values[bottom_coords] = np.full(sum(bottom_coords), 1)
    return values
kappa2 = Function(Q)
kappa2.interpolate(eval_kappa)

In [6]:
# Define variational formulation and b.c.
V = functionspace(mesh, ("Lagrange", 1))
u, v = TrialFunction(V), TestFunction(V)
a = inner(kappa*grad(u), grad(v))*dx
x = SpatialCoordinate(mesh)
L = Constant(mesh, default_scalar_type(1))*v*dx
dofs = locate_dofs_geometrical(V, lambda x: np.isclose(x[0], 0))
bcs = [dirichletbc(default_scalar_type(1), dofs, V)]

In [7]:
# Solve problem
problem = LinearProblem(
    a, L, bcs = bcs, 
    petsc_options={"ksp_type": "preonly", "pc_type": "lu"}, petsc_options_prefix="subdomains_structured_",
    )
uh = problem.solve()

# Filter out ghosted cells
tdim = mesh.topology.dim
num_cells_local = mesh.topology.index_map(tdim).size_local
marker = np.zeros(num_cells_local, dtype=np.int32)
cells_0 = cells_0[cells_0 < num_cells_local]
cells_1 = cells_1[cells_1 < num_cells_local]
marker[cells_0] = 1
marker[cells_1] = 2
mesh.topology.create_connectivity(tdim, tdim)
p = pyvista.Plotter(window_size=[800,800])
grid = pyvista.UnstructuredGrid(*vtk_mesh(mesh, tdim, np.arange(num_cells_local, dtype=np.int32)))
grid.cell_data["Marker"] = marker
grid.set_active_scalars("Marker")
p.add_mesh(grid, show_edges=True)
p.view_xy()
if not pyvista.OFF_SCREEN:
    p.show()

2026-03-10 14:14:10.477 (   0.474s) [    7762CFD71600]vtkXOpenGLRenderWindow.:1458  WARN| bad X server connection. DISPLAY=


Widget(value='<iframe src="http://localhost:43313/index.html?ui=P_0x7761b4f2f110_0&reconnect=auto" class="pyvi…

In [8]:
from visualization_fct import plotScalarFunction
plotScalarFunction(V, uh)

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

 There is also more to this tutorial where they generate the subdomains with gmsh: https://jsdokken.com/dolfinx-tutorial/chapter3/subdomains.html